# 04 · Evaluation & Artifact Export

| Step | Description |
|---|---|
| **Evaluation** | Consolidated metrics table · per-model comparison chart · diagnostic plots |
| **Export** | Processed Parquet · model artifacts · experiment YAML log |
| **Dashboard prep** | Verifies all files required by `Home.py` are present |

**Reads:** `processed/*.parquet` · `processed/metrics_draft.yaml` · `processed/split_meta.yaml`  
**Saves:** `reports/metrics/forecast_metrics.csv` · `configs/experiments.yaml` (appended)

In [ ]:
## Setup

import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ.pop("SPARK_HOME", None)   # Let pyspark 4.2.0 use its own bundled JARs

import sys, warnings, yaml, joblib
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.experiment_tracker import log_run

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED = PROJECT_ROOT / CFG['artifacts']['processed_dir']
MODELS    = PROJECT_ROOT / CFG['artifacts']['models_dir']
FIGURES   = PROJECT_ROOT / CFG['reports']['figures_dir']
METRICS   = PROJECT_ROOT / CFG['reports']['metrics_dir']
METRICS.mkdir(parents=True, exist_ok=True)

with open(PROCESSED / 'split_meta.yaml') as f:
    SPLIT = yaml.safe_load(f)

FEATURE_COLS = SPLIT['feature_cols']
TARGET_COL   = SPLIT['target_col']
TS_COL       = SPLIT['ts_col']
TRAIN_END    = SPLIT['train_end']
VAL_END      = SPLIT['val_end']
TEST_END     = SPLIT['test_end']

print('Setup complete.')

---
## 1 · Consolidated Metrics Table

In [ ]:
# ── Load metrics collected by notebook 03 ────────────────────────────
with open(PROCESSED / 'metrics_draft.yaml') as f:
    results_raw = yaml.safe_load(f)

results_df = pd.DataFrame(results_raw)
results_df = results_df.sort_values(['partition','mae']).reset_index(drop=True)

# Save to metrics dir
metrics_csv = METRICS / 'forecast_metrics.csv'
results_df.to_csv(metrics_csv, index=False)
print(f'Metrics saved → {metrics_csv}')

# Display styled table
results_df.style.format({'mae':'{:.6f}','rmse':'{:.6f}','smape':'{:.2f}%'}) \
                .background_gradient(subset=['mae','rmse'], cmap='RdYlGn_r') \
                .set_caption('Model Evaluation — All Partitions')

In [ ]:
# ── Test-set comparison bar chart ────────────────────────────────────
test_res = results_df[results_df['partition'] == 'Test']

fig = go.Figure()
for metric, col in [('mae','#4FC3F7'), ('rmse','#F7931A')]:
    fig.add_trace(go.Bar(x=test_res['model'], y=test_res[metric],
                         name=metric.upper(), marker_color=col))
fig.update_layout(barmode='group', title='Test-Set Metrics Comparison',
                  template='plotly_dark', yaxis_title='Error')
fig.write_html(str(FIGURES / 'metrics_comparison.html'))
fig.show()

In [ ]:
# ── sMAPE comparison ─────────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=test_res['model'], y=test_res['smape'],
    marker_color='#66BB6A', name='sMAPE %',
))
fig.update_layout(title='Test-Set sMAPE Comparison',
                  template='plotly_dark', yaxis_title='sMAPE (%)')
fig.write_html(str(FIGURES / 'smape_comparison.html'))
fig.show()

---
## 2 · Forecast Diagnostic Plots

In [ ]:
# ── Residual analysis for best ML model ──────────────────────────────
test_df = pd.read_parquet(PROCESSED / 'test.parquet')
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

best_model_path = MODELS / 'lgbm_return.pkl'
if best_model_path.exists():
    art = joblib.load(best_model_path)
    preds     = art['model'].predict(art['scaler'].transform(X_test))
    residuals = y_test.values - preds

    fig = go.Figure([
        go.Scatter(x=pd.to_datetime(test_df[TS_COL]), y=residuals,
                   mode='lines', name='Residual', line=dict(color='#F7931A', width=0.8)),
        go.Scatter(x=pd.to_datetime(test_df[TS_COL]),
                   y=[0]*len(residuals), name='Zero', line=dict(color='#aaa', dash='dot')),
    ])
    fig.update_layout(title='LightGBM — Test Residuals', template='plotly_dark',
                      yaxis_title='Actual − Predicted')
    fig.write_html(str(FIGURES / 'lgbm_residuals.html'))
    fig.show()

    # Residual histogram
    fig2 = go.Figure(go.Histogram(x=residuals, nbinsx=100, marker_color='#4FC3F7',
                                  histnorm='probability density'))
    fig2.update_layout(title='LightGBM — Residual Distribution',
                       template='plotly_dark', xaxis_title='Residual')
    fig2.write_html(str(FIGURES / 'lgbm_residual_dist.html'))
    fig2.show()
else:
    print(f'⚠️  {best_model_path} not found. Run notebook 03 first.')

---
## 3 · Artifact Export

In [ ]:
# ── Verify processed feature Parquet exists (created by notebook 02) ─
features_path = PROCESSED / 'btc_features.parquet'
if features_path.exists():
    df_feat = pd.read_parquet(features_path)
    print(f'btc_features.parquet : {df_feat.shape}  ✅')
else:
    print('⚠️  btc_features.parquet not found — re-run notebook 02.')

In [ ]:
# ── Log all runs to configs/experiments.yaml ──────────────────────────
for row in results_raw:
    if row['partition'] == 'Test':
        artifact_p = str(MODELS / f'{row["model"].lower()}_return.pkl')
        log_run(
            model         = row['model'],
            target        = TARGET_COL,
            train_end     = TRAIN_END,
            val_end       = VAL_END,
            test_end      = TEST_END,
            metrics       = {k: row[k] for k in ('mae','rmse','smape')},
            artifact_path = artifact_p if Path(artifact_p).exists() else 'n/a',
        )
print('All runs logged to configs/experiments.yaml.')

---
## 4 · Dashboard Readiness Check

In [ ]:
# ── Verify all files the Streamlit app expects ────────────────────────
REQUIRED_FOR_APP = [
    PROCESSED / 'btc_features.parquet',
    MODELS    / 'lgbm_return.pkl',
    MODELS    / 'catboost_return.pkl',
    MODELS    / 'gmm.pkl',
    MODELS    / 'hmm.pkl',
    MODELS    / 'isolation_forest.pkl',
    MODELS    / 'anomaly_scores.csv',
    MODELS    / 'changepoints.csv',
    MODELS    / 'garch_cond_vol.csv',
    METRICS   / 'forecast_metrics.csv',
]

print('Dashboard artifact checklist:')
all_ready = True
for p in REQUIRED_FOR_APP:
    status = '✅' if Path(p).exists() else '❌  MISSING'
    if '❌' in status:
        all_ready = False
    print(f'  {status}  {p.name}')

print()
if all_ready:
    print('All artifacts ready. Launch the dashboard with:')
    print('  streamlit run Home.py')
else:
    print('Some artifacts are missing. Re-run the relevant notebook section.')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────
print('=' * 60)
print('  PIPELINE COMPLETE')
print('=' * 60)
print(f'  Processed data : {PROCESSED}')
print(f'  Model artifacts: {MODELS}')
print(f'  Figures        : {FIGURES}')
print(f'  Metrics        : {METRICS}')
print()
print('Best test-set results:')
best = results_df[results_df['partition']=='Test'].sort_values('mae').head(3)
print(best[['model','mae','rmse','smape']].to_string(index=False))